In [1]:
from transformers import AutoTokenizer, AutoProcessor, SiglipTextModel, SiglipVisionModel

siglip_text_model = SiglipTextModel.from_pretrained("google/siglip-base-patch16-256")
#siglip_text_model.to(device)
siglip_text_model.to('cuda:1')
siglip_vision_model = SiglipVisionModel.from_pretrained("google/siglip-base-patch16-256")
#siglip_vision_model.to(device)
siglip_vision_model.to('cuda:1')
siglip_tokenizer = AutoTokenizer.from_pretrained("google/siglip-base-patch16-256")
siglip_processor = AutoProcessor.from_pretrained("google/siglip-base-patch16-256")

config.json:   0%|          | 0.00/322 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/813M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/711 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/798k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/368 [00:00<?, ?B/s]

In [2]:
!pip install transformers
!pip install git+https://github.com/openai/CLIP.git
!pip install --upgrade google-api-core>=2.10.2,<3.0.0dev
!pip install clipcap
!pip install open_clip_torch==2.32.0
!pip install fairscale
!pip install -r requirements.txt -q
!pip uninstall -y timm
!pip install timm==0.4.12
!pip install tensorboardX
!pip install torchscale
!pip install sacremoses

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-rnu7natr
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-rnu7natr
  Resolved https://github.com/openai/CLIP.git to commit dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.8 MB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369489 sha256=192f8de9a722857955d88a8fd91c8e5d66a335ec325209a96b2f6f00195a2f44
  Stored in directory: /tmp/pip-ephem-wheel-cache-3_ubvn8j/wheels/da/2b/4c/d6691fa9597aac8bb85d2ac13b112deb897d5b50f5ad9a37e4
Successfully built clip
/bin/bash: line 1: 3.0.0dev: No such file or directory
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 26.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.3/266.3 kB 6.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel .

In [3]:
import open_clip
import torch
from transformers import (GPT2Tokenizer, GPT2LMHeadModel)
from transformers import CLIPTokenizer
import torch
import clip
import requests
from PIL import Image
import os
from torch import nn
import numpy as np
import torch.nn.functional as nnf
from typing import Tuple, List, Union, Optional, Callable
from transformers import GPT2Tokenizer, GPT2LMHeadModel, get_linear_schedule_with_warmup
from tqdm import tqdm, trange
import skimage.io as io
import PIL.Image
from IPython.display import Image 
from copy import deepcopy
import torchvision.transforms as transforms 
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib as mpl
import torchvision.datasets as datasets
from torchvision.transforms.functional import InterpolationMode
import imageio
from skimage.transform import resize
from transformers import pipeline, set_seed
from kaggle_secrets import UserSecretsClient
import torchvision.transforms as transforms
import glob
import torch
import cv2
from PIL import Image
from tqdm import tqdm
import sys
from transformers import AlignProcessor, AlignModel
import torchvision.transforms.functional as TF
import torchvision.transforms as T
from types import MethodType
import torch.nn.functional as F
from transformers.image_utils import ChannelDimension
import random

In [4]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

cuda:0


In [5]:
def crop_text(text: str, max_len: int = 85) -> str:
    if len(text) <= max_len:
        return text
    return text[:max_len]

def crop_string_by_tokens(text: str, max_tokens: int = 25) -> str:
    tokens = text.split()  # tách theo whitespace
    if len(tokens) <= max_tokens:
        return text
    return " ".join(tokens[:max_tokens])

In [6]:
align_processor = AlignProcessor.from_pretrained("kakaobrain/align-base")
align_model = AlignModel.from_pretrained("kakaobrain/align-base")
#align_model.to(device)
align_model.to('cuda:1')

preprocessor_config.json:   0%|          | 0.00/508 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/399 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/690M [00:00<?, ?B/s]

AlignModel(
  (text_model): AlignTextModel(
    (embeddings): AlignTextEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): AlignTextEncoder(
      (layer): ModuleList(
        (0-11): 12 x AlignTextLayer(
          (attention): AlignTextAttention(
            (self): AlignTextSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): AlignTextSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerN

In [7]:
!git clone https://github.com/fonzi22/unilm.git /kaggle/working/BEIT3

import sys
sys.path.append('/kaggle/working/BEIT3/beit3')

Cloning into '/kaggle/working/BEIT3'...
remote: Enumerating objects: 10069, done.
remote: Total 10069 (delta 0), reused 0 (delta 0), pack-reused 10069 (from 1)
Receiving objects: 100% (10069/10069), 65.40 MiB | 35.51 MiB/s, done.
Resolving deltas: 100% (4733/4733), done.


In [8]:
import modeling_finetune
from torchvision import transforms
from torchvision.datasets.folder import default_loader
from transformers import XLMRobertaTokenizer
from timm import create_model
from timm.data.constants import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD, IMAGENET_INCEPTION_MEAN, IMAGENET_INCEPTION_STD
from IPython.display import clear_output

In [9]:
checkpoint = torch.load('/kaggle/input/beit3-weights/beit3_large_patch16_384_coco_retrieval.pth')
beit3_model = create_model('beit3_large_patch16_384_retrieval')
beit3_model.load_state_dict(checkpoint['model'])
#beit3_model.eval().to(device)
beit3_model.eval().to('cuda:0')
beit3_tokenizer = XLMRobertaTokenizer('/kaggle/input/beit3-spm/beit3.spm')
clear_output()

In [10]:
!pip install salesforce-lavis
#!pip install --upgrade transformers
from lavis.models import load_model_and_preprocess

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.4/235.4 kB 6.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Using cached timm-0.4.12-py3-none-any.whl.metadata (30 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.3/100.3 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.8/47.8 MB 41.4 MB/s eta 0:00:00
Using cached timm-0.4.12-py3-none-any.whl (376 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 114.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 94.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.4/226.4 kB 20.8 MB/s et

In [11]:
albef_model, albef_vis_processors, albef_txt_processors = load_model_and_preprocess(name="albef_feature_extractor", model_type="base", is_eval=False, device='cuda:1')
albef_model.to('cuda:1')

Downloading: "https://dl.fbaipublicfiles.com/deit/deit_base_patch16_224-b5f2ef4d.pth" to /root/.cache/torch/hub/checkpoints/deit_base_patch16_224-b5f2ef4d.pth
100%|██████████| 330M/330M [00:01<00:00, 190MB/s]


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

100%|██████████| 3.25G/3.25G [00:13<00:00, 269MB/s]
/usr/local/lib/python3.10/dist-packages/lavis/models/albef_models/__init__.py:35: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feat

reshape position embedding from 256 to 196


AlbefFeatureExtractor(
  (visual_encoder): VisionTransformerEncoder(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      (norm): Identity()
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (blocks): ModuleList(
      (0-11): 12 x Block(
        (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=768, out_features=2304, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=768, out_features=768, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (drop_path): Identity()
        (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): Mlp(
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (act): GELU(approximate='none')
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (drop): Dropout(p=0.0, inplace

In [12]:
import torch.nn.functional as F
from lavis.common.registry import registry
from lavis.common.utils import get_abs_path
from lavis.models.albef_models import AlbefBase
from lavis.models.albef_models.albef_outputs import AlbefOutputFeatures
from lavis.models.med import BertForMaskedLM
from lavis.models.vit import VisionTransformerEncoder
from torch import nn
from transformers import BertConfig
import warnings

def custom_extract_features(self, samples, mode="multimodal"):
    # Copy the original extract_features code without @torch.no_grad()
    image = samples["image"]
    caption = samples["text_input"]
    if isinstance(mode, str):
        mode = [mode]
    for m in mode:
        assert m in ["multimodal", "image", "text"], f"Mode must be one of [multimodal, image, text], got {m}"
    image_embeds, text_embeds, multimodal_embeds = None, None, None
    image_features, text_features = None, None
    if "image" in mode or "multimodal" in mode:
        assert image is not None, "Image must be provided for 'image' or 'multimodal' mode"
        image_embeds = self.visual_encoder.forward_features(image)
        image_features = F.normalize(self.vision_proj(image_embeds), dim=-1)
    if "text" in mode or "multimodal" in mode:
        assert caption is not None, "Text must be provided for 'text' or 'multimodal' mode"
        text = self.tokenizer(caption, padding=True, return_tensors="pt").to(self.device)
        text_output = self.text_encoder.bert(
            text.input_ids, attention_mask=text.attention_mask, return_dict=True, mode="text"
        )
        text_embeds = text_output.last_hidden_state
        text_features = F.normalize(self.text_proj(text_embeds), dim=-1)
    if "multimodal" in mode:
        image_atts = torch.ones(image_embeds.size()[:-1], dtype=torch.long).to(self.device)
        output = self.text_encoder.bert(
            encoder_embeds=text_embeds,
            attention_mask=text.attention_mask,
            encoder_hidden_states=image_embeds,
            encoder_attention_mask=image_atts,
            return_dict=True,
            mode="fusion",
        )
        multimodal_embeds = output.last_hidden_state
    return AlbefOutputFeatures(
        image_embeds=image_embeds,
        image_embeds_proj=image_features,
        text_embeds=text_embeds,
        text_embeds_proj=text_features,
        multimodal_embeds=multimodal_embeds,
    )

albef_model.extract_features = MethodType(custom_extract_features, albef_model)

In [13]:
import sys
!git clone https://github.com/salesforce/BLIP.git /kaggle/working/BLIP

sys.path.append('/kaggle/working/BLIP')

Cloning into '/kaggle/working/BLIP'...
remote: Enumerating objects: 277, done.
remote: Total 277 (delta 0), reused 0 (delta 0), pack-reused 277 (from 1)
Receiving objects: 100% (277/277), 7.16 MiB | 25.37 MiB/s, done.
Resolving deltas: 100% (151/151), done.


In [14]:
class MLP(nn.Module):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)
    def __init__(self, sizes: Tuple[int, ...], bias=True, act=nn.Tanh):
        super(MLP, self).__init__()
        layers = []
        for i in range(len(sizes)-1):
            layers.append(nn.Linear(sizes[i], sizes[i+1], bias=bias))
            if i < len(sizes) - 2:
                layers.append(act())
        self.model = nn.Sequential(*layers)

class ClipCaptionModel(nn.Module):
    def get_dummy_token(self, batch_size: int, device: torch.device) -> torch.Tensor:
        return torch.zeros(batch_size, self.prefix_length, dtype=torch.int64, device=device)
        
    def forward(self, tokens: torch.Tensor, prefix: torch.Tensor, mask: Optional[torch.Tensor] = None, labels: Optional[torch.Tensor] = None):
        embedding_text = self.gpt.transformer.wte(tokens)
        prefix_projections = self.clip_project(prefix).view(-1, self.prefix_length, self.gpt_embedding_size)
        embedding_cat = torch.cat((prefix_projections, embedding_text), dim=1)
        if labels is not None:
            dummy_token = self.get_dummy_token(tokens.shape[0], tokens.device)
            labels = torch.cat((dummy_token, tokens), dim=1)
        out = self.gpt(inputs_embeds=embedding_cat, labels=labels, attention_mask=mask)
        return out
        
    def __init__(self, prefix_length: int, prefix_size: int = 512):
        super(ClipCaptionModel, self).__init__()
        self.prefix_length = prefix_length
        self.gpt = GPT2LMHeadModel.from_pretrained('gpt2')
        self.gpt_embedding_size = self.gpt.transformer.wte.weight.shape[1]
        if prefix_length > 10: 
            self.clip_project = nn.Linear(prefix_size, self.gpt_embedding_size * prefix_length)
        else:
            self.clip_project = MLP((prefix_size, (self.gpt_embedding_size*prefix_length) //2 , self.gpt_embedding_size * prefix_length))

class ClipCaptionPrefix(ClipCaptionModel):
    def parameters(self, recurse: bool = True):
        return self.clip_project.parameters()

    def train(self, mode: bool=True):
        super(ClipCaptionPrefix, self).train(mode)
        self.gpt.eval()
        return self
        
def generate_beam(model, tokenizer, beam_size: int=5, prompt=None, embed=None, entry_length=67, temperature=1., stop_token: str = '.'):
    model.eval()
    stop_token_index = tokenizer.encode(stop_token)[0]
    tokens = None
    scores = None
    device = next(model.parameters()).device
    # seq_lengths = torch.ones(beam_size, device=device)
    seq_lengths = torch.ones(beam_size, device='cuda:1')
    # is_stopped = torch.zeros(beam_size, device=device, dtype=torch.bool)
    is_stopped = torch.zeros(beam_size, device='cuda:1', dtype=torch.bool)
    with torch.no_grad():
        if embed is None:
            generated = embed
        else:
            if token is None:
                tokens = torch.tensor(tokenizer.encode(prompt))
                # tokens = tokens.unsqueeze(0).to(device)
                tokens = tokens.unsqueeze(0).to('cuda:1')
                generated = model.gpt.transformer.wte(tokens)

        for i in range(entry_length):
            outputs = model.gpt(inputs_embeds=generated)
            logits = outputs.logits
            logits = logits[:, -1, :] /  (temperature if temperature > 0 else 1.0)
            logits = logits.softmax(-1).log()
            if scores is None:
                scores, next_tokens = logits.topk(beam_size, -1)
                generated = generated.expand(beam_size, *generated.shape[1:])
                next_tokens, scores = next_tokens.permute(1, 0), scores.squeeze(0)
                if tokens is None:
                    tokens = next_tokens
                else:
                    tokens = tokens.expand(beam_size, *tokens.shape[1:])
                    tokens = torch.cat((tokens, next_tokens), dim=1)

            else:
                logits[is_stopped] = -float(np.inf)
                logits[is_stopped, 0] = 0
                scores_sum = scores[:, None] + logits
                seq_lengths[~is_stopped] += 1
                scores_sum_average = scores_sum / seq_lengths[:, None]
                scores_sum_average, next_tokens = scores_sum_average.view(-1).topk(beam_size, -1)
                next_tokens_source = next_tokens // scores_sum.shape[1]
                seq_lengths = seq_lengths[next_tokens_source]
                next_tokens = next_tokens % scores_sum.shape[1]
                next_tokens = next_tokens.unsqueeze(1)
                tokens = tokens[next_tokens_source]
                tokens = torch.cat((tokens, next_tokens), dim = 1)
                generated = generated[next_tokens_source]
                scores = scores_sum_average*seq_lengths
                is_stopped = is_stopped[next_tokens_source]

            next_token_embed = model.gpt.transformer.wte(next_tokens.squeeze()).view(generated.shape[0], 1, -1)
            generated = torch.cat((generated, next_token_embed), dim = 1)
            is_stopped = is_stopped + next_tokens_eq(stop_token_index).squeeze()
            if is_stopped.all():
                break
    scores = scores/seq_lengths
    output_list = tokens.cpu().numpy()
    output_texts = [tokenizer.decode(output[:int(length)]) for output, length in zip(output_list, seq_lengths)]
    order = scores.argsort(descending=True)
    output_texts = [output_texts[i] for i in order]
    return output_texts
def generate2(model, tokenizer, tokens=None, prompt=None, embed=None, entry_count=1, entry_length=67, top_p=0.8, temperature=1., stop_token: str = '.'):
    model.eval()
    generated_num = 0
    generated_list = []
    stop_token_index = tokenizer.encode(stop_token)[0]
    filter_value = -float("Inf")
    device = next(model.parameters()).device

    with torch.no_grad():
        for entry_idx in range(entry_count):
            if embed is not None:
                generated = embed
            else:
                if tokens is None:
                    tokens = torch.tensor(tokenizer.encode(prompt))
                    # tokens = tokens.unsqueeze(0).to(device)
                    tokens = tokens.unsqueeze(0).to('cuda:1')

                generated = model.gpt.transformer.wte(tokens)

            for i in range(entry_length):
                outputs = model.gpt(inputs_embeds=generated)
                logits = outputs.logits
                logits = logits[:, -1, :] / (temperature if temperature > 0 else 1.0)
                sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                cumulative_probs = torch.cumsum(nnf.softmax(sorted_logits, dim=-1), dim=-1)
                sorted_indices_to_remove = cumulative_probs > top_p
                sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                sorted_indices_to_remove[..., 0] = 0

                indices_to_remove = sorted_indices[sorted_indices_to_remove]
                logits[:, indices_to_remove] = filter_value
                next_token = torch.argmax(logits, -1).unsqueeze(0)
                next_token_embed = model.gpt.transformer.wte(next_token)
                if tokens is None:
                    tokens = next_token
                else:
                    tokens = torch.cat((tokens, next_token), dim=1)
                generated = torch.cat((generated, next_token_embed), dim=1)
                if stop_token_index == next_token.item():
                    break
            output_list = list(tokens.squeeze().cpu().numpy())
            output_text = tokenizer.decode(output_list)
            generated_list.append(output_text)
            
    return generated_list[0]

In [15]:
model_path = '/kaggle/input/clip-model/model_wieghts.pt'

In [16]:
# clip_vit_B32_model, preprocess = clip.load("ViT-B/32", device=device, jit=False)
clip_vit_B32_model, preprocess = clip.load("ViT-B/32", device='cuda:1', jit=False)
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

prefix_length = 10

caption_model = ClipCaptionModel(prefix_length)
state_dict = torch.load(model_path, map_location=torch.device('cpu'), weights_only=True)
caption_model.load_state_dict(state_dict, strict=False)

caption_model = caption_model.eval()
#caption_model = caption_model.to(device)
caption_model = caption_model.to('cuda:1')

100%|████████████████████████████████████████| 338M/338M [00:01<00:00, 259MiB/s]


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [17]:
from models.blip import blip_decoder
from models.blip import blip_feature_extractor

captioning_model_path = '/kaggle/input/another-checkpoint/model_base_capfilt_large (1).pth'
med_config_path = '/kaggle/working/BLIP/configs/med_config.json'

blip_model_captioning = blip_decoder(pretrained=captioning_model_path, med_config = med_config_path, image_size=384, vit='base')
blip_model_captioning.eval()
#blip_model_captioning.to(device)
blip_model_captioning.to('cuda:0')


feature_extractor_model_path = '/kaggle/input/feature-extracting-model/model_base.pth'
blip_model_feature_extractor = blip_feature_extractor(pretrained=feature_extractor_model_path, med_config = med_config_path, image_size=384, vit='base')
blip_model_feature_extractor.eval()
#blip_model_feature_extractor.to(device)
blip_model_feature_extractor.to('cuda:0')

reshape position embedding from 196 to 576
load checkpoint from /kaggle/input/another-checkpoint/model_base_capfilt_large (1).pth
reshape position embedding from 196 to 576
load checkpoint from /kaggle/input/feature-extracting-model/model_base.pth


BLIP_Base(
  (visual_encoder): VisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      (norm): Identity()
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (blocks): ModuleList(
      (0-11): 12 x Block(
        (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=768, out_features=2304, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=768, out_features=768, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (drop_path): Identity()
        (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): Mlp(
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (act): GELU(approximate='none')
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (drop): Dropout(p=0.0, inplace=False)
        )
 

In [18]:
def blip(image, blip_model_captioning=blip_model_captioning, image_size=384):
    image = image.unsqueeze(0)
    blip_model_captioning.eval()
    #blip_model_captioning = blip_model_captioning.to(device)
    blip_model_captioning = blip_model_captioning.to('cuda:0')

    with torch.no_grad():
        # beam search
        caption = blip_model_captioning.generate(image, sample=False, num_beams=1, max_length=50, min_length=5)
        #caption = model.generate(image, sample=True, num_beams=5, max_length=50, min_length=5)
    return caption[0]

In [19]:
#clip_rn_101_model, preprocess_rn_101 = clip.load("RN101", device=device, jit=False)
clip_rn_101_model, preprocess_rn_101 = clip.load("RN101", device='cuda:1', jit=False)
clip_rn_101_model.eval()
#clip_rn_101_model.to(device)
clip_rn_101_model.to('cuda:1')

#clip_vit_B16_model, preprocess_vit_B16 = clip.load("ViT-B/16", device=device, jit=False)
clip_vit_B16_model, preprocess_vit_B16 = clip.load("ViT-B/16", device='cuda:1', jit=False)
clip_vit_B16_model.eval()
#clip_vit_B16_model.to(device)
clip_vit_B16_model.to('cuda:1')

#clip_vit_L14_model, preprocess_vit_L14 = clip.load("ViT-L/14", device=device, jit=False)
clip_vit_L14_model, preprocess_vit_L14 = clip.load("ViT-L/14", device='cuda:0', jit=False)
clip_vit_L14_model.eval()
#clip_vit_L14_model.to(device)
clip_vit_L14_model.to('cuda:0')

100%|████████████████████████████████████████| 278M/278M [00:01<00:00, 219MiB/s]
100%|███████████████████████████████████████| 335M/335M [00:05<00:00, 69.4MiB/s]
100%|███████████████████████████████████████| 890M/890M [00:12<00:00, 75.7MiB/s]


CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
    (ln_pre): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): Sequential(
        (0): ResidualAttentionBlock(
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=1024, out_features=1024, bias=True)
          )
          (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=1024, out_features=4096, bias=True)
            (gelu): QuickGELU()
            (c_proj): Linear(in_features=4096, out_features=1024, bias=True)
          )
          (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        )
        (1): ResidualAttentionBlock(
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=1024, out_features=1024, bias=True)


In [20]:
coca_model, _, coca_transform = open_clip.create_model_and_transforms(model_name="coca_ViT-L-14", pretrained="mscoco_finetuned_laion2B-s13B-b90k")
#coca_model = coca_model.to(device)
coca_model = coca_model.to('cuda:1')
coca_tokenizer = open_clip.get_tokenizer("coca_ViT-L-14")

open_clip_pytorch_model.bin:   0%|          | 0.00/2.55G [00:00<?, ?B/s]

In [21]:
import shutil
from transformers import utils as transformers_utils

In [22]:
transformers_modules = [key for key in sys.modules if 'transformers' in key]
for module_name in transformers_modules:
    sys.modules.pop(module_name, None)

cache_dir = os.getenv("TRANSFORMERS_CACHE", os.path.expanduser("~/.cache/huggingface"))
if os.path.exists(cache_dir):
    try:
        shutil.rmtree(cache_dir)
        print(f"Cleared transformers cache at {cache_dir}")
    except Exception as e:
        print(f"Error clearing cache: {e}")
else:
    print(f"No cache directory found at {cache_dir}")

Cleared transformers cache at /root/.cache/huggingface


In [23]:
#!pip install -U transformers

In [24]:
import shutil
from transformers import utils as transformers_utils

transformers_modules = [key for key in sys.modules if 'transformers' in key]
for module_name in transformers_modules:
    sys.modules.pop(module_name, None)

cache_dir = os.getenv("TRANSFORMERS_CACHE", os.path.expanduser("~/.cache/huggingface"))
if os.path.exists(cache_dir):
    try:
        shutil.rmtree(cache_dir)
        print(f"Cleared transformers cache at {cache_dir}")
    except Exception as e:
        print(f"Error clearing cache: {e}")
else:
    print(f"No cache directory found at {cache_dir}")

Cleared transformers cache at /root/.cache/huggingface


In [25]:
!pip uninstall transformers -y
!git clone --single-branch --branch feature/add_transformers https://github.com/OFA-Sys/OFA.git
!pip install OFA/transformers/
!git lfs install
!git clone https://huggingface.co/OFA-Sys/OFA-base

Found existing installation: transformers 4.26.1
Uninstalling transformers-4.26.1:
  Successfully uninstalled transformers-4.26.1
Cloning into 'OFA'...
remote: Enumerating objects: 5745, done.
remote: Counting objects: 100% (932/932), done.
remote: Compressing objects: 100% (256/256), done.
remote: Total 5745 (delta 710), reused 676 (delta 676), pack-reused 4813 (from 1)
Receiving objects: 100% (5745/5745), 97.78 MiB | 47.50 MiB/s, done.
Resolving deltas: 100% (2243/2243), done.
Processing ./OFA/transformers
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-4.18.0.dev0-py3-none-any.whl size=3916872 sha256=7ecc7bf21060e9314cb9a215f8db926ce772d4b44f1575e147d4c7448a9d4de7
  Stored in directory: /tmp/pip-ephem-wheel-cache-aex80c9d/wheels/0e/73/4e/28100b384ff1a7f2ec69547e2b93faff478bb6acc175edbbf1
Successfully built transformers
ERROR: pip's dependency

In [26]:
from transformers import OFATokenizer, OFAModel
from transformers.models.ofa.generate import sequence_generator

ckpt_dir = "/kaggle/working/OFA-base"

ofa_tokenizer = OFATokenizer.from_pretrained(ckpt_dir)
ofa_model = OFAModel.from_pretrained(ckpt_dir, use_cache=False).to('cuda:0')

/kaggle/working/OFA-base
<super: <class 'OFATokenizer'>, <OFATokenizer object>>


In [27]:
ofa_mean, ofa_std = [0.5, 0.5, 0.5], [0.5, 0.5, 0.5]
ofa_resolution = 384
ofa_patch_resize_transform = transforms.Compose([
    transforms.Resize((ofa_resolution, ofa_resolution)),
    transforms.Lambda(lambda x: x.float() / 255.0),
    transforms.Normalize(mean=ofa_mean, std=ofa_std),
])

ofa_txt = "describe the image"
ofa_inputs = ofa_tokenizer([ofa_txt], return_tensors="pt").input_ids
ofa_generator = sequence_generator.SequenceGenerator(
        tokenizer=ofa_tokenizer, beam_size=5, max_len_b=16, min_len=0, no_repeat_ngram_size=3)

def ofa_caption_tensor_image(image_tensor):
    patch_img = ofa_patch_resize_transform(image_tensor).unsqueeze(0)
    data = {}
    data["net_input"] = {"input_ids": ofa_inputs.to('cuda:0'), 'patch_images': patch_img.to('cuda:0'), 'patch_masks':torch.tensor([True])}
    gen_output = ofa_generator.generate([ofa_model], data)
    gen = [gen_output[i][0]["tokens"] for i in range(len(gen_output))]
    caption = ofa_tokenizer.batch_decode(gen, skip_special_tokens=True)
    return caption

In [28]:
torch.manual_seed(42)
np.random.seed(42)
torch.cuda.manual_seed_all(42)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
class AdversarialCaptionAttack:
    def __init__(self, caption_model, tokenizer, clip_vit_B32_model, clip_vit_B16_model, clip_vit_L14_model, clip_rn_101_model, 
                 albef_model, albef_txt_processors, 
                 #albef_vis_processors,
                 blip_model_captioning, blip_model_feature_extractor, 
                 coca_model, coca_tokenizer,
                 #coca_transform, 
                 beit3_model, beit3_tokenizer, 
                 align_model, align_processor, 
                 siglip_tokenizer, siglip_vision_model, siglip_text_model, 
                 #siglip_processor, 
                 device, 
                 alpha: float = 0.7, epsilon: float = 16.0 / 255.0, pgd_steps: int = 40, beta: float = 0.8, step_size: float = 4.0 / 255.0):
        self.caption_model = caption_model
        self.tokenizer = tokenizer
        self.clip_vit_B32_model = clip_vit_B32_model
        self.clip_vit_B16_model = clip_vit_B16_model
        self.clip_vit_L14_model = clip_vit_L14_model
        self.clip_rn_101_model = clip_rn_101_model
        self.albef_model = albef_model
        #self.albef_vis_processors = albef_vis_processors
        self.albef_txt_processors = albef_txt_processors
        self.albef_tensor_processors = None

        self.albef_model.eval()  
        self.albef_model.zero_grad(set_to_none=True)  
        
        self.blip_model_captioning = blip_model_captioning
        self.blip_model_feature_extractor = blip_model_feature_extractor
        self.coca_model = coca_model
        self.coca_tokenizer = coca_tokenizer
        self.coca_transform_for_tensor = None
        #self.coca_transform = coca_transform
        self.beit3_model = beit3_model
        self.beit3_tokenizer = beit3_tokenizer
        self.align_model = align_model
        self.align_processor = align_processor
        self.siglip_tokenizer = siglip_tokenizer
        #self.siglip_processor = siglip_processor
        self.siglip_vision_model = siglip_vision_model
        self.siglip_text_model = siglip_text_model
        self.device = device
        self.alpha = alpha
        self.epsilon = epsilon
        self.pgd_steps = pgd_steps
        self.beta = beta
        self.cosine_similarity = nn.CosineSimilarity(dim=1, eps=1e-6)
        self.step_size = step_size


    def tensor_siglip_processor(self, image_tensor, size=(256, 256), resample='bicubic', do_rescale=True, rescale_factor=1/255, do_normalize=True, 
                    image_mean=[0.5, 0.5, 0.5], image_std=[0.5, 0.5, 0.5], data_format=ChannelDimension.FIRST):
    
        if image_tensor.shape[0] != 3:
            raise ValueError(f"Expected 3 channels, got shape {image_tensor.shape}")
        if image_tensor.dtype != torch.float32:
            image_tensor = image_tensor.to(torch.float32)

        min_val, max_val = image_tensor.min().item(), image_tensor.max().item()
        if max_val <= 1.1 and min_val >= -0.1:
            image_tensor = image_tensor * 255
        elif min_val < 0 or max_val > 255.1:
            print(f"Warning: Input range [{min_val}, {max_val}] not in [0, 255]")

        image_tensor = image_tensor.requires_grad_(True)

        resized_image = F.interpolate(
            image_tensor.unsqueeze(0),
            size=size,
            mode=resample,
            align_corners=False,
            antialias=True  # Smoother resizing
        ).squeeze(0)

        processed_image = resized_image
        if do_rescale:
            processed_image = processed_image * rescale_factor

        if do_normalize:
            mean = torch.tensor(image_mean, device=processed_image.device).view(3, 1, 1)
            std = torch.tensor(image_std, device=processed_image.device).view(3, 1, 1)
            processed_image = (processed_image - mean) / std

        if data_format == ChannelDimension.LAST:
            processed_image = processed_image.permute(1, 2, 0)

        return processed_image
    
    def tensor_align_processor(self, image_tensor, size=(346, 346), crop_size=(289, 289), do_center_crop=True, do_rescale=True,
        rescale_factor=0.00784313725490196, rescale_offset=True, do_normalize=False, image_mean=[0.5, 0.5, 0.5], image_std=[0.5, 0.5, 0.5], data_format=ChannelDimension.FIRST):

        if image_tensor.shape[0] != 3:
            raise ValueError(f"Expected 3 channels, got shape {image_tensor.shape}")
        if image_tensor.dtype != torch.float32:
            print(f"Warning: Converting dtype from {image_tensor.dtype} to float32")
            image_tensor = image_tensor.to(torch.float32)

        min_val, max_val = image_tensor.min().item(), image_tensor.max().item()
        if max_val <= 1.1 and min_val >= -0.1:  # Likely [0, 1]
            #print("Input range appears to be [0, 1], scaling to [0, 255]")
            image_tensor = image_tensor * 255
        elif min_val < 0 or max_val > 255.1:
            print(f"Warning: Input range [{min_val}, {max_val}] not in [0, 255]")

        image_tensor = image_tensor.requires_grad_(True)

        resized_image = F.interpolate(
            image_tensor.unsqueeze(0),
            size=size,
            mode='bilinear',
            align_corners=False
        ).squeeze(0)
        #print(f"After resize to {size}: min={resized_image.min().item()}, max={resized_image.max().item()}")

        if do_center_crop:
            _, h, w = resized_image.shape
            crop_height, crop_width = crop_size
            top = (h - crop_height) // 2
            left = (w - crop_width) // 2

            if h < crop_height or w < crop_width:
                pad_top = max(0, (crop_height - h) // 2)
                pad_bottom = max(0, crop_height - h - pad_top)
                pad_left = max(0, (crop_width - w) // 2)
                pad_right = max(0, crop_width - w - pad_left)
                resized_image = F.pad(
                    resized_image,
                    (pad_left, pad_right, pad_top, pad_bottom),
                    mode='constant',
                    value=0
                )
                _, h, w = resized_image.shape
                top = (h - crop_height) // 2
                left = (w - crop_width) // 2

            processed_image = resized_image[:, top:top + crop_height, left:left + crop_width]
            #print(f"After crop to {crop_size}: min={processed_image.min().item()}, max={processed_image.max().item()}")
        else:
            processed_image = resized_image

        # 3. Rescale
        if do_rescale:
            processed_image = processed_image * rescale_factor
            #print(f"After rescale (*{rescale_factor}): min={processed_image.min().item()}, max={processed_image.max().item()}")
            if rescale_offset:
                processed_image = processed_image - 1
                #print(f"After offset (-1): min={processed_image.min().item()}, max={processed_image.max().item()}")

        if do_normalize:
            mean = torch.tensor(image_mean, device=processed_image.device).view(3, 1, 1)
            std = torch.tensor(image_std, device=processed_image.device).view(3, 1, 1)
            processed_image = (processed_image - mean) / std

        if data_format == ChannelDimension.LAST:
            processed_image = processed_image.permute(1, 2, 0)

        return processed_image
    
    def tensor_coca_transform(self, image_size: Union[int, Tuple[int, int]] = 224,
        mean: Tuple[float, float, float] = (0.48145466, 0.4578275, 0.40821073), std: Tuple[float, float, float] = (0.26862954, 0.26130258, 0.27577711),
        interpolation: str = 'bicubic') -> callable:

        if isinstance(image_size, int):
            image_size = (image_size, image_size)
        
        target_h, target_w = image_size

        interpolation_modes = {
            'bilinear': TF.InterpolationMode.BILINEAR,
            'bicubic': TF.InterpolationMode.BICUBIC,
            'nearest': TF.InterpolationMode.NEAREST,
        }
        interpolation_mode = interpolation_modes.get(interpolation, TF.InterpolationMode.BICUBIC)

        def transform(tensor: torch.Tensor) -> torch.Tensor:
            if tensor.ndim != 3:
                raise ValueError(f"Expected 3D tensor (C, H, W), got shape {tensor.shape}")
            if tensor.shape[0] not in (1, 3, 4):
                raise ValueError(f"Expected 1, 3, or 4 channels, got {tensor.shape[0]} channels")

            if tensor.dtype == torch.uint8:
                tensor = tensor.float() / 255.0
            elif tensor.dtype in (torch.float32, torch.float16):
                if tensor.max() > 1.0 + 1e-5 or tensor.min() < 0.0 - 1e-5:
                    raise ValueError("Float tensor values must be in [0, 1]")
            else:
                raise ValueError(f"Unsupported tensor dtype: {tensor.dtype}")

            h, w = tensor.shape[1], tensor.shape[2]
            scale = max(target_h / h, target_w / w)  # Shortest-side resizing
            new_h, new_w = int(h * scale), int(w * scale)
            tensor = TF.resize(tensor, size=[new_h, new_w], interpolation=interpolation_mode)

            # Center Crop to target size
            tensor = TF.center_crop(tensor, output_size=[target_h, target_w])

            # Normalize
            tensor = TF.normalize(tensor, mean=mean, std=std)

            return tensor
        
        return transform
    

    def tensor_albef_vis_processor(self, image_size: Union[int, Tuple[int, int]] = 224, mean: Tuple[float, float, float] = (0.48145466, 0.4578275, 0.40821073),
                                    std: Tuple[float, float, float] = (0.26862954, 0.26130258, 0.27577711), interpolation: str = 'bicubic') -> Callable:

        if isinstance(image_size, int):
            image_size = (image_size, image_size)
        target_h, target_w = image_size

        # Map interpolation string to torchvision enum
        interpolation_modes = {
            'bilinear': TF.InterpolationMode.BILINEAR,
            'bicubic': TF.InterpolationMode.BICUBIC,
            'nearest': TF.InterpolationMode.NEAREST,
        }
        interpolation_mode = interpolation_modes.get(interpolation, TF.InterpolationMode.BICUBIC)

        def transform(tensor: torch.Tensor) -> torch.Tensor:
            if tensor.ndim != 3:
                raise ValueError(f"Expected 3D tensor (C, H, W), got shape {tensor.shape}")
            if tensor.shape[0] not in (1, 3, 4):
                raise ValueError(f"Expected 1, 3, or 4 channels, got {tensor.shape[0]} channels")

            if tensor.dtype == torch.uint8:
                tensor = tensor.float() / 255.0
            elif tensor.dtype in (torch.float32, torch.float16):
                if tensor.max() > 1.0 + 1e-5 or tensor.min() < 0.0 - 1e-5:
                    raise ValueError("Float tensor values must be in [0, 1]")
            else:
                raise ValueError(f"Unsupported tensor dtype: {tensor.dtype}")

            #Resize to target size
            tensor = TF.resize(tensor, size=[target_h, target_w], interpolation=interpolation_mode)

            # Normalize
            tensor = TF.normalize(tensor, mean=mean, std=std)

            return tensor

        return transform
    
    def generate_caption_clipcap(self, image: torch.Tensor) -> str:
        with torch.no_grad():  
            prefix = self.clip_vit_B32_model.encode_image(image.to('cuda:1')).to('cuda:1', dtype=torch.float32)
            prefix_embed = self.caption_model.clip_project(prefix).reshape(1, self.caption_model.prefix_length, -1)
            caption = generate2(self.caption_model, self.tokenizer, embed=prefix_embed)
        return caption
    
    def generate_caption_blip(self, image: torch.Tensor) -> str:
        # kỳ vọng image là [1, 3, 224, 224]
        with torch.no_grad():
            resize_384 = transforms.Resize((384, 384), interpolation=transforms.InterpolationMode.BILINEAR)
            image_clone = image.clone()
            image_clone = resize_384(image_clone)
            image_clone = image_clone.squeeze()

            caption = blip(image = image_clone.to("cuda:0"), blip_model_captioning=self.blip_model_captioning, image_size=384)    
        return caption
    
    def generate_caption_coca(self, image: torch.Tensor) -> str:
        #to_pil = transforms.ToPILImage()
        #pil_image = to_pil(image.squeeze())
        #img_for_coca = coca_transform(pil_image).unsqueeze(0)

        self.coca_transform_for_tensor = self.tensor_coca_transform(
            image_size=224,
            mean=(0.48145466, 0.4578275, 0.40821073),
            std=(0.26862954, 0.26130258, 0.27577711),
            interpolation='bicubic'
        )

        img_for_coca = self.coca_transform_for_tensor(image.squeeze()).unsqueeze(0)
        # img_for_coca = img_for_coca.to('cuda:1')

        with torch.no_grad(), torch.cuda.amp.autocast():
            generated = self.coca_model.generate(img_for_coca.to('cuda:1'))
            caption = open_clip.decode(generated[0]).split("<end_of_text>")[0].replace("<start_of_text>", "")
        return caption
    
    def get_embedding_clip_vit_b32(self, image: torch.Tensor=None, text: str = None) -> torch.Tensor:
        if image is not None:
            embedding = self.clip_vit_B32_model.encode_image(image.to('cuda:1')).to('cuda:1', dtype=torch.float32)
            embedding = embedding / embedding.norm(dim=1, keepdim=True)
            #embedding = self.caption_model.clip_project(embedding).reshape(1, prefix_length, -1)
            #embedding = embedding.mean(dim=1)  
            return embedding
        if text is not None:
            tokens = clip.tokenize([text]).to('cuda:1')
            embedding = self.clip_vit_B32_model.encode_text(tokens).to('cuda:1', dtype=torch.float32)
            embedding = embedding / embedding.norm(dim=1, keepdim=True)

            #embedding = self.caption_model.clip_project(embedding).reshape(1, prefix_length, -1)
            #embedding = embedding.mean(dim=1)  
            return embedding
    
    def get_embedding_clip_rn_101(self, image: torch.Tensor=None, text: str = None) -> torch.Tensor:
        if image is not None:
            embedding = self.clip_rn_101_model.encode_image(image.to('cuda:1')).to('cuda:1', dtype=torch.float32)
            embedding = embedding / embedding.norm(dim=1, keepdim=True)

            return embedding
            
        if text is not None:
            tokens = clip.tokenize([text]).to('cuda:1')
            embedding = self.clip_rn_101_model.encode_text(tokens).to('cuda:1', dtype=torch.float32)
            embedding = embedding / embedding.norm(dim=1, keepdim=True)

            return embedding

    def get_embedding_clip_vit_b16(self, image: torch.Tensor=None, text: str = None) -> torch.Tensor:
        if image is not None:
            embedding = self.clip_vit_B16_model.encode_image(image.to('cuda:1')).to('cuda:1', dtype=torch.float32)
            embedding = embedding / embedding.norm(dim=1, keepdim=True)
            return embedding
            
        if text is not None:
            tokens = clip.tokenize([text]).to('cuda:1')
            embedding = self.clip_vit_B16_model.encode_text(tokens).to('cuda:1', dtype=torch.float32)
            embedding = embedding / embedding.norm(dim=1, keepdim=True)
            return embedding

    def get_embedding_clip_vit_l14(self, image: torch.Tensor=None, text: str = None) -> torch.Tensor:
        if image is not None:
            embedding = self.clip_vit_L14_model.encode_image(image.to('cuda:0')).to('cuda:0', dtype=torch.float32)
            embedding = embedding / embedding.norm(dim=1, keepdim=True)

            #del image
            torch.cuda.empty_cache()
            
            return embedding
            
        if text is not None:
            tokens = clip.tokenize([text]).to('cuda:0')
            embedding = self.clip_vit_L14_model.encode_text(tokens).to('cuda:0', dtype=torch.float32)
            embedding = embedding / embedding.norm(dim=1, keepdim=True)
            return embedding
    
    def get_embedding_albef(self, image: torch.Tensor=None, text: str = None) -> torch.Tensor:
        if image is not None:
            #to_pil = transforms.ToPILImage()
            #pil_image = to_pil(image.squeeze())
            
            self.albef_tensor_processors = self.tensor_albef_vis_processor(
                image_size=224,
                mean=(0.48145466, 0.4578275, 0.40821073),
                std=(0.26862954, 0.26130258, 0.27577711),
                interpolation='bicubic')

            image_for_albef = self.albef_tensor_processors(image.squeeze()).unsqueeze(0).to('cuda:1')
            #image_for_albef = self.albef_vis_processors["eval"](pil_image).unsqueeze(0).to('cuda:1') 
            image_for_albef = {"image": image_for_albef, "text_input": None}
            embedding = self.albef_model.extract_features(image_for_albef, mode="image")
            embedding = embedding.image_embeds_proj[:, 0, :]
            embedding = embedding / embedding.norm(dim=1, keepdim=True)
            return embedding
        
        if text is not None:
            text_for_albef = self.albef_txt_processors["eval"](text)
            text_for_albef = {"image": None, "text_input": [text_for_albef]}
            embedding = self.albef_model.extract_features(text_for_albef, mode="text")
            embedding = embedding.text_embeds_proj[:, 0, :]
            embedding = embedding / embedding.norm(dim=1, keepdim=True)
            return embedding
    
    def get_embedding_blip(self, image: torch.Tensor=None, text: str = None) -> torch.Tensor:
        # kỳ vọng image là [1, 3, 224, 224]
        if image is not None:

            resize_384 = transforms.Resize((384, 384), interpolation=transforms.InterpolationMode.BILINEAR)
            image_clone = image.clone()
            image_clone = resize_384(image_clone)

            embedding = self.blip_model_feature_extractor(image_clone.to('cuda:0'), "", mode='image')[0, 0]
            embedding = embedding.unsqueeze(0)
            embedding = embedding / embedding.norm(dim=1, keepdim=True)

            # del resize_384, image_clone
            torch.cuda.empty_cache()

            return embedding
            
        if text is not None:
            embedding = self.blip_model_feature_extractor(torch.empty(1, 3, 384, 384).to('cuda:0'), text, mode='text')[0,0]
            embedding = embedding.unsqueeze(0)
            embedding = embedding / embedding.norm(dim=1, keepdim=True)

            return embedding
    
    def get_embedding_coca(self, image: torch.Tensor=None, text: str = None) -> torch.Tensor:
        if image is not None:
            #to_pil = transforms.ToPILImage()
            #pil_image = to_pil(image.squeeze())
            #image_for_coca = self.coca_transform(pil_image).unsqueeze(0).to('cuda:1')

            self.coca_transform_for_tensor = self.tensor_coca_transform(
                image_size=224,
                mean=(0.48145466, 0.4578275, 0.40821073),
                std=(0.26862954, 0.26130258, 0.27577711),
                interpolation='bicubic'
            )
            
            image_for_coca = self.coca_transform_for_tensor(image.squeeze()).unsqueeze(0).to('cuda:1')    
            embedding = self.coca_model.encode_image(image_for_coca)
            #del image_for_coca  
            #torch.cuda.empty_cache()
            embedding = embedding / embedding.norm(dim=1, keepdim=True)
            return embedding

        if text is not None:
            text_for_coca = self.coca_tokenizer([text]).to('cuda:1')
            embedding = self.coca_model.encode_text(text_for_coca)
            embedding = embedding / embedding.norm(dim=1, keepdim=True)
            return embedding
    
    def get_embedding_beit3(self, image: torch.Tensor=None, text: str = None) -> torch.Tensor:
        if image is not None:
            resize_384 = transforms.Resize((384, 384), interpolation=transforms.InterpolationMode.BILINEAR)
            image_clone = image.clone()
            image_clone = resize_384(image_clone)

            embedding = self.beit3_model(image=image_clone.to('cuda:0'), only_infer=True)[0]
            embedding = embedding / embedding.norm(dim=1, keepdim=True)

            # del resize_384, image_clone
            torch.cuda.empty_cache()
            
            return embedding

        if text is not None:
            def beit3_get_text_segment(text, tokenizer, max_len = 64):
                tokens = tokenizer.tokenize(text)
                tokens = tokenizer.convert_tokens_to_ids(tokens)
                if len(tokens) > max_len -2:
                    tokens = tokens[:max_len-2]
                tokens = [tokenizer.bos_token_id] + tokens[:] + [tokenizer.eos_token_id]
                num_tokens = len(tokens)
                padding_mask = [0] * num_tokens + [1] * (max_len - num_tokens)
                language_tokens = tokens + [tokenizer.pad_token_id] * (max_len - num_tokens)
                return torch.tensor([language_tokens]),  torch.tensor([padding_mask]), num_tokens

            def encode_text_query_beit3(query, model, tokenizer):
                language_tokens, padding_mask, _ = beit3_get_text_segment(query, tokenizer)
                language_tokens = language_tokens.to(device)
                padding_mask = padding_mask.to(device)
                _, language_cls = model(text_description=language_tokens, padding_mask=padding_mask, only_infer=True)

                result = language_cls
                return result
            
            embedding = encode_text_query_beit3(text, self.beit3_model, self.beit3_tokenizer)
            embedding = embedding / embedding.norm(dim=1, keepdim=True)

            return embedding
    
    def get_embedding_align(self, image: torch.Tensor=None, text: str=None) -> torch.Tensor:
        if image is not None:
            #to_pil = transforms.ToPILImage()
            #pil_image = to_pil(image.squeeze())

            #image_input_align = self.align_processor(images=pil_image, return_tensors="pt").to('cuda:1')
            #embedding = self.align_model.get_image_features(pixel_values=image_input_align['pixel_values'])
            image_input_align = self.tensor_align_processor(image.squeeze().to('cuda:1'), size=(self.align_processor.image_processor.size['height'], self.align_processor.image_processor.size['width']),
                crop_size=(self.align_processor.image_processor.crop_size['height'], self.align_processor.image_processor.crop_size['width']), do_center_crop=self.align_processor.image_processor.do_center_crop, do_rescale=self.align_processor.image_processor.do_rescale,
                rescale_factor=self.align_processor.image_processor.rescale_factor, rescale_offset=self.align_processor.image_processor.rescale_offset, do_normalize=self.align_processor.image_processor.do_normalize, image_mean=self.align_processor.image_processor.image_mean,
                image_std=self.align_processor.image_processor.image_std, data_format=ChannelDimension.FIRST)
            
            embedding = self.align_model.get_image_features(pixel_values=image_input_align.unsqueeze(0))

            #del image_input_align
            #torch.cuda.empty_cache()
            
            embedding = embedding / embedding.norm(dim=1, keepdim=True)
            return embedding



        if text is not None:
            text_input_align = self.align_processor(text=text, return_tensors="pt").to('cuda:1')
            embedding = self.align_model.get_text_features(input_ids=text_input_align['input_ids'], 
                                                           attention_mask=text_input_align['attention_mask'], 
                                                           token_type_ids=text_input_align['token_type_ids'])

            #del text_input_align
            #torch.cuda.empty_cache()  
            
            embedding = embedding / embedding.norm(dim=1, keepdim=True)
            return embedding
    
    def get_embedding_siglip(self, image: torch.Tensor=None, text: str=None) -> torch.Tensor:
        if image is not None:
            #to_pil = transforms.ToPILImage()
            #pil_image = to_pil(image.squeeze())
            #image_input_siglip = self.siglip_processor(images=pil_image, return_tensors="pt")
            #image_input_siglip = image_input_siglip.to('cuda:1')

            image_input_siglip = self.tensor_siglip_processor(image.squeeze().to('cuda:1'), size=(256, 256),
                resample='bicubic', do_rescale=True, rescale_factor=1/255, do_normalize=True, image_mean=[0.5, 0.5, 0.5], image_std=[0.5, 0.5, 0.5], data_format=ChannelDimension.FIRST)
            #image_output_siglip = self.siglip_vision_model(**image_input_siglip)
            image_output_siglip = self.siglip_vision_model(pixel_values=image_input_siglip.unsqueeze(0))

            #del image_input_siglip
            #torch.cuda.empty_cache()  
            
            embedding = image_output_siglip.pooler_output
            embedding = embedding / embedding.norm(dim=1, keepdim=True)
            return embedding
            
        if text is not None:
            text_input_siglip = self.siglip_tokenizer([text], padding="max_length", return_tensors="pt")
            text_input_siglip = text_input_siglip.to('cuda:1')
            text_output_siglip = self.siglip_text_model(**text_input_siglip)
            embedding = text_output_siglip.pooler_output
            embedding = embedding / embedding.norm(dim=1, keepdim=True)
            return embedding
    
    def get_joint_embedding(self, image_embedding: torch.Tensor, text_embedding: torch.Tensor) -> torch.Tensor:
        joint = self.alpha * image_embedding + (1 - self.alpha) * text_embedding
        return joint/joint.norm(dim=1, keepdim=True)

    def get_full_embedding(self, ori_pil_image, ori_caption, target_pil_image, target_caption):
        ori_tensor_image = T.ToTensor()(ori_pil_image)
        target_tensor_image = T.ToTensor()(target_pil_image)

        if ori_tensor_image.ndim != 4:
            ori_tensor_image = ori_tensor_image.unsqueeze(0)
        if target_tensor_image.ndim != 4:
            target_tensor_image = target_tensor_image.unsqueeze(0)

        embeddings = []
        for model_name, get_embedding in [
            ("clip_vit_b32", self.get_embedding_clip_vit_b32),
            ("clip_rn_101", self.get_embedding_clip_rn_101),
            ("clip_vit_b16", self.get_embedding_clip_vit_b16),
            ("clip_vit_l14", self.get_embedding_clip_vit_l14),
            ("albef", self.get_embedding_albef),
            ("blip", self.get_embedding_blip),
            ("coca", self.get_embedding_coca),
            ("beit3", self.get_embedding_beit3),
            ("align", self.get_embedding_align),
            ("siglip", self.get_embedding_siglip)
        ]:
            with torch.no_grad():
                embeddings.append(get_embedding(image=ori_tensor_image).to("cpu"))
                embeddings.append(get_embedding(text=ori_caption).to("cpu"))
                embeddings.append(get_embedding(image=target_tensor_image).to("cpu"))
                embeddings.append(get_embedding(text=target_caption).to("cpu"))
            torch.cuda.empty_cache()  
        return embeddings
        
    def attack(self, 
               
               target_image_embedding_clip_vit_b32: torch.Tensor, target_text_embedding_clip_vit_b32: torch.Tensor, 
               ori_image_embedding_clip_vit_b32: torch.Tensor, ori_text_embedding_clip_vit_b32: torch.Tensor, 

               target_image_embedding_clip_rn_101: torch.Tensor, target_text_embedding_clip_rn_101: torch.Tensor,
               ori_image_embedding_clip_rn_101: torch.Tensor, ori_text_embedding_clip_rn_101: torch.Tensor,

               target_image_embedding_clip_vit_b16: torch.Tensor, target_text_embedding_clip_vit_b16: torch.Tensor, 
               ori_image_embedding_clip_vit_b16: torch.Tensor, ori_text_embedding_clip_vit_b16: torch.Tensor, 

               target_image_embedding_clip_vit_l14: torch.Tensor, target_text_embedding_clip_vit_l14: torch.Tensor, 
               ori_image_embedding_clip_vit_l14: torch.Tensor, ori_text_embedding_clip_vit_l14: torch.Tensor, 

               target_image_embedding_albef: torch.Tensor, target_text_embedding_albef: torch.Tensor,
               ori_image_embedding_albef: torch.Tensor, ori_text_embedding_albef: torch.Tensor,

               target_image_embedding_blip: torch.Tensor, target_text_embedding_blip: torch.Tensor,
               ori_image_embedding_blip: torch.Tensor, ori_text_embedding_blip: torch.Tensor,

               target_image_embedding_coca: torch.Tensor, target_text_embedding_coca: torch.Tensor,
               ori_image_embedding_coca: torch.Tensor, ori_text_embedding_coca: torch.Tensor,

               target_image_embedding_beit3: torch.Tensor, target_text_embedding_beit3: torch.Tensor,
               ori_image_embedding_beit3: torch.Tensor, ori_text_embedding_beit3: torch.Tensor,

               target_image_embedding_align: torch.Tensor, target_text_embedding_align: torch.Tensor,
               ori_image_embedding_align: torch.Tensor, ori_text_embedding_align: torch.Tensor,

               target_image_embedding_siglip: torch.Tensor, target_text_embedding_siglip: torch.Tensor,
               ori_image_embedding_siglip: torch.Tensor, ori_text_embedding_siglip: torch.Tensor,
               
               ori_image: torch.Tensor, tgt_image: torch.Tensor, binary_list=[1, 0, 0, 0, 0, 0, 0, 0, 0, 0]) -> torch.Tensor: 

        model_configs = [
            ("clip_vit_b32", 0, ori_image_embedding_clip_vit_b32, ori_text_embedding_clip_vit_b32, 
             target_image_embedding_clip_vit_b32, target_text_embedding_clip_vit_b32),
            ("clip_rn_101", 1, ori_image_embedding_clip_rn_101, ori_text_embedding_clip_rn_101, 
             target_image_embedding_clip_rn_101, target_text_embedding_clip_rn_101),
            ("clip_vit_b16", 2, ori_image_embedding_clip_vit_b16, ori_text_embedding_clip_vit_b16, 
             target_image_embedding_clip_vit_b16, target_text_embedding_clip_vit_b16),
            ("clip_vit_l14", 3, ori_image_embedding_clip_vit_l14, ori_text_embedding_clip_vit_l14, 
             target_image_embedding_clip_vit_l14, target_text_embedding_clip_vit_l14),
            ("albef", 4, ori_image_embedding_albef, ori_text_embedding_albef, 
             target_image_embedding_albef, target_text_embedding_albef),
            ("blip", 5, ori_image_embedding_blip, ori_text_embedding_blip, 
             target_image_embedding_blip, target_text_embedding_blip),
            ("coca", 6, ori_image_embedding_coca, ori_text_embedding_coca, 
             target_image_embedding_coca, target_text_embedding_coca),
            ("beit3", 7, ori_image_embedding_beit3, ori_text_embedding_beit3, 
             target_image_embedding_beit3, target_text_embedding_beit3),
            ("align", 8, ori_image_embedding_align, ori_text_embedding_align, 
             target_image_embedding_align, target_text_embedding_align),
            ("siglip", 9, ori_image_embedding_siglip, ori_text_embedding_siglip, 
             target_image_embedding_siglip, target_text_embedding_siglip),
            ]

        ori_joint = {}
        targeted_joint = {}
        
        for model_name, idx, ori_img_emb, ori_txt_emb, tgt_img_emb, tgt_txt_emb in model_configs:
            if binary_list[idx] == 1:
                ori_joint[model_name] = self.get_joint_embedding(ori_img_emb, ori_txt_emb)
                targeted_joint[model_name] = self.get_joint_embedding(tgt_img_emb, tgt_txt_emb)
        
        ori_image = ori_image.clone().detach().to(self.device)
        noise_generator = torch.Generator(device=self.device).manual_seed(42)
        delta = torch.zeros_like(ori_image).uniform_(-self.epsilon, self.epsilon, generator=noise_generator).to(self.device)
        del noise_generator
        delta.requires_grad_(True)

        best_loss = float('inf')
        best_adv_image = None
        best_delta = None

        for step in range(self.pgd_steps):
            adv_image = ori_image + delta
            adv_image = torch.clamp(adv_image, 0, 1)

            def input_diversity(x, p=0.7, min_ratio=0.8, max_ratio=1.2):
                if torch.rand(1).item() > p:
                    return x
                B, C, H, W = x.shape
                ratio = float(torch.empty(1).uniform_(min_ratio, max_ratio))
                newH, newW = int(H * ratio), int(W * ratio)

                x2 = F.interpolate(x, size=(newH, newW), mode='bilinear', align_corners=False)
                padH = max(0, H - newH)
                padW = max(0, W - newW)

                pad_top = torch.randint(0, padH + 1, (1,)).item() if padH > 0 else 0
                pad_left = torch.randint(0, padW + 1, (1,)).item() if padW > 0 else 0
                pad_bottom = padH - pad_top
                pad_right = padW - pad_left

                x2 = F.pad(x2, (pad_left, pad_right, pad_top, pad_bottom))
                x2 = x2[:, :, :H, :W]

                return x2   

            adv_image_aug = input_diversity(adv_image, min_ratio=0.8, max_ratio=1.2)
            
            with torch.cuda.amp.autocast():
                adv_embeddings = {}
                if binary_list[0] == 1:
                    adv_embeddings["clip_vit_b32"] = self.get_embedding_clip_vit_b32(image=adv_image_aug)
                if binary_list[1] == 1:
                    adv_embeddings["clip_rn_101"] = self.get_embedding_clip_rn_101(image=adv_image_aug)
                if binary_list[2] == 1:
                    adv_embeddings["clip_vit_b16"] = self.get_embedding_clip_vit_b16(image=adv_image_aug)
                if binary_list[3] == 1:
                    adv_embeddings["clip_vit_l14"] = self.get_embedding_clip_vit_l14(image=adv_image_aug)
                if binary_list[4] == 1:
                    adv_embeddings["albef"] = self.get_embedding_albef(image=adv_image_aug)
                if binary_list[5] == 1:
                    adv_embeddings["blip"] = self.get_embedding_blip(image=adv_image_aug)
                if binary_list[6] == 1:
                    adv_embeddings["coca"] = self.get_embedding_coca(image=adv_image_aug)
                if binary_list[7] == 1:
                    adv_embeddings["beit3"] = self.get_embedding_beit3(image=adv_image_aug)
                if binary_list[8] == 1:
                    adv_embeddings["align"] = self.get_embedding_align(image=adv_image_aug)
                if binary_list[9] == 1:
                    adv_embeddings["siglip"] = self.get_embedding_siglip(image=adv_image_aug)

                current_caption_clipcap = self.generate_caption_clipcap(adv_image) if binary_list[0] == 1 else None
                current_caption_blip = self.generate_caption_blip(adv_image) if binary_list[5] == 1 else None
                current_caption_coca = self.generate_caption_coca(adv_image) if binary_list[6] == 1 else None

                current_text_embedding_clipcap = self.get_embedding_clip_vit_b32(text=current_caption_clipcap) if binary_list[0] == 1 else None
                current_text_embedding_blip = self.get_embedding_blip(text=current_caption_blip) if binary_list[5] == 1 else None
                current_text_embedding_coca = self.get_embedding_coca(text=current_caption_coca) if binary_list[6] == 1 else None

                current_adv_joint = {}
                if binary_list[0] == 1:
                    current_adv_joint["clip_vit_b32"] = self.get_joint_embedding(adv_embeddings["clip_vit_b32"], current_text_embedding_clipcap)
                if binary_list[5] == 1:
                    current_adv_joint["blip"] = self.get_joint_embedding(adv_embeddings["blip"], current_text_embedding_blip)
                if binary_list[6] == 1:
                    current_adv_joint["coca"] = self.get_joint_embedding(adv_embeddings["coca"], current_text_embedding_coca)

                ori_sim = 0.0
                targeted_sim = 0.0
                for model_name, idx, _, _, _, _ in model_configs:
                    if binary_list[idx] == 1:
                        device = 'cuda:0' if model_name in ["clip_vit_l14", "blip", "beit3"] else 'cuda:1'
                        if model_name in ["clip_vit_b32", "blip", "coca"]:
                            ori_sim += torch.mean(torch.sum(current_adv_joint[model_name].to(device) * ori_joint[model_name].to(device), dim=1)).to(self.device)
                            targeted_sim += torch.mean(torch.sum(current_adv_joint[model_name].to(device) * targeted_joint[model_name].to(device), dim=1)).to(self.device)

                        else:
                            ori_sim += torch.mean(torch.sum(adv_embeddings[model_name].to(device) * ori_joint[model_name].to(device), dim=1)).to(self.device)
                            targeted_sim += torch.mean(torch.sum(adv_embeddings[model_name].to(device) * targeted_joint[model_name].to(device), dim=1)).to(self.device)
                            
            loss = - targeted_sim + self.beta * ori_sim 

            current_loss = loss.item()
            if current_loss < best_loss:
                best_loss = current_loss
                best_adv_image = adv_image.clone().detach()
                #print(f"  New best loss found: {best_loss:.4f}")

            loss.backward()

            grad = delta.grad.detach()
            delta_update = self.step_size * torch.sign(grad)
            delta.data = torch.clamp(delta - delta_update, min=-self.epsilon, max=self.epsilon)

            delta.grad = None  
            loss = None
            
            #print(f"Step {step}:")
            #if binary_list[0] == 1:
            #    print(f"  Current caption clipcap: {current_caption_clipcap}")
            #if binary_list[5] == 1:
            #    print(f"  Current caption blip: {current_caption_blip}")
            #if binary_list[6] == 1:
            #    print(f"  Current caption coca: {current_caption_coca}")
            #print(f"  Original similarity: {ori_sim.item():.4f}")
            #print(f"  Targeted similarity: {targeted_sim.item():.4f}")
            #print(f"  Loss: {current_loss:.4f}")
            #print(f"  Delta norm: {delta.norm().item():.4f}")
            #print(f"  Gradient norm: {grad.norm().item():.4f}")
            #print(f"  Delta update norm: {delta_update.norm().item():.4f}")

            #delta.grad.zero_()
            del adv_embeddings, current_caption_clipcap, current_caption_blip, current_caption_coca
            del current_text_embedding_clipcap, current_text_embedding_blip, current_text_embedding_coca
            del current_adv_joint
            del ori_sim, targeted_sim
            gc.collect()
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

        if best_adv_image is None:
            best_adv_image = torch.clamp(ori_image + delta, 0, 1)  
            best_delta = delta.clone().detach()
        print(f"Best loss achieved: {best_loss:.4f}")
        
        del model_configs, ori_joint, targeted_joint, delta, loss#, ori_image, best_loss
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        
        return best_adv_image, best_delta

In [ ]:
class Solution:
    def __init__(self, binary_list):
        self.binary_list = binary_list
        self.clip_rn_101_score = 0
        self.clip_vit_b16_score = 0
        self.clip_vit_b32_score = 0
        self.clip_vit_l14_score = 0
        self.unified_metric = 0

def evaluate_one_solution(binary_list):
    adv_image, _ = attack_model.attack(
    
    target_image_embedding_clip_vit_b32=target_image_embedding_clip_vit_b32, target_text_embedding_clip_vit_b32=target_text_embedding_clip_vit_b32, 
    ori_image_embedding_clip_vit_b32=ori_image_embedding_clip_vit_b32, ori_text_embedding_clip_vit_b32=ori_text_embedding_clip_vit_b32, 

    target_image_embedding_clip_rn_101=target_image_embedding_clip_rn_101, target_text_embedding_clip_rn_101=target_text_embedding_clip_rn_101,
    ori_image_embedding_clip_rn_101=ori_image_embedding_clip_rn_101, ori_text_embedding_clip_rn_101=ori_text_embedding_clip_rn_101,

    target_image_embedding_clip_vit_b16=target_image_embedding_clip_vit_b16, target_text_embedding_clip_vit_b16=target_text_embedding_clip_vit_b16, 
    ori_image_embedding_clip_vit_b16=ori_image_embedding_clip_vit_b16, ori_text_embedding_clip_vit_b16=ori_text_embedding_clip_vit_b16, 

    target_image_embedding_clip_vit_l14=target_image_embedding_clip_vit_l14, target_text_embedding_clip_vit_l14=target_text_embedding_clip_vit_l14, 
    ori_image_embedding_clip_vit_l14=ori_image_embedding_clip_vit_l14, ori_text_embedding_clip_vit_l14=ori_text_embedding_clip_vit_l14, 

    target_image_embedding_albef = target_image_embedding_albef, target_text_embedding_albef = target_text_embedding_albef,
    ori_image_embedding_albef = ori_image_embedding_albef, ori_text_embedding_albef = ori_text_embedding_albef,

    target_image_embedding_blip=target_image_embedding_blip, target_text_embedding_blip=target_text_embedding_blip,
    ori_image_embedding_blip=ori_image_embedding_blip, ori_text_embedding_blip=ori_text_embedding_blip,

    target_image_embedding_coca= target_image_embedding_coca, target_text_embedding_coca = target_text_embedding_coca,
    ori_image_embedding_coca= target_image_embedding_coca, ori_text_embedding_coca = ori_text_embedding_coca,

    target_image_embedding_beit3=target_image_embedding_beit3, target_text_embedding_beit3=target_text_embedding_beit3,
    ori_image_embedding_beit3=ori_image_embedding_beit3, ori_text_embedding_beit3=ori_text_embedding_beit3,

    target_image_embedding_align=target_image_embedding_align, target_text_embedding_align=target_text_embedding_align,
    ori_image_embedding_align=ori_image_embedding_align, ori_text_embedding_align=ori_text_embedding_align,

    target_image_embedding_siglip=target_image_embedding_siglip, target_text_embedding_siglip=target_text_embedding_siglip,
    ori_image_embedding_siglip=ori_image_embedding_siglip, ori_text_embedding_siglip=ori_text_embedding_siglip,
        
    ori_image=ori_image_tensor, tgt_image=tgt_image_tensor, binary_list=binary_list)

    img = adv_image.detach().cpu()  
    if img.shape[0] == 1:
        img = img.squeeze(0)
    img = img.permute(1, 2, 0) 
    img = (img.numpy() * 255).astype(np.uint8)  
    pil_img = Image.fromarray(img)

    preprocess_image = transforms.Compose([
    transforms.ToTensor()])

    pil_img = preprocess_image(pil_img)
    pil_img = (pil_img * 255).to(torch.uint8)
    caption = ofa_caption_tensor_image(pil_img)
    caption = caption[0]
    caption = trim_string(caption)
    print("caption from targeted model: ", caption)
    del adv_image

    torch.cuda.empty_cache()
    gc.collect()
        
    with torch.no_grad():
        tokens = clip.tokenize([caption])

        adv_clip_rn_101_embedding = clip_rn_101_model.encode_text(tokens.to("cuda:1")).to("cpu")
        adv_clip_vit_b16_embedding = clip_vit_B16_model.encode_text(tokens.to("cuda:1")).to("cpu")
        adv_clip_vit_b32_embedding = clip_vit_B32_model.encode_text(tokens.to("cuda:1")).to("cpu")
        adv_clip_vit_l14_embedding = clip_vit_L14_model.encode_text(tokens.to("cuda:0")).to("cpu")

    del tokens
    torch.cuda.empty_cache()

    adv_clip_rn_101_embedding = adv_clip_rn_101_embedding / adv_clip_rn_101_embedding.norm(dim=1, keepdim=True)
    adv_clip_vit_b16_embedding = adv_clip_vit_b16_embedding / adv_clip_vit_b16_embedding.norm(dim=1, keepdim=True)
    adv_clip_vit_b32_embedding = adv_clip_vit_b32_embedding / adv_clip_vit_b32_embedding.norm(dim=1, keepdim=True)
    adv_clip_vit_l14_embedding = adv_clip_vit_l14_embedding / adv_clip_vit_l14_embedding.norm(dim=1, keepdim=True)

    clip_rn_101_score = torch.mean(torch.sum(adv_clip_rn_101_embedding * tgt_clip_rn_101_embedding, dim=1))
    del adv_clip_rn_101_embedding
    torch.cuda.empty_cache()

    clip_vit_b16_score = torch.mean(torch.sum(adv_clip_vit_b16_embedding * tgt_clip_vit_b16_embedding, dim=1))
    del adv_clip_vit_b16_embedding
    torch.cuda.empty_cache()

    clip_vit_b32_score = torch.mean(torch.sum(adv_clip_vit_b32_embedding * tgt_clip_vit_b32_embedding, dim=1))
    del adv_clip_vit_b32_embedding
    torch.cuda.empty_cache()

    clip_vit_l14_score = torch.mean(torch.sum(adv_clip_vit_l14_embedding * tgt_clip_vit_l14_embedding, dim=1))
    del adv_clip_vit_l14_embedding
    torch.cuda.empty_cache()

    return clip_rn_101_score, clip_vit_b16_score, clip_vit_b32_score, clip_vit_l14_score

In [33]:
import gc
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [34]:
attack_model = AdversarialCaptionAttack(caption_model=caption_model, tokenizer=tokenizer,         
        clip_vit_B32_model=clip_vit_B32_model, clip_rn_101_model=clip_rn_101_model,
        clip_vit_B16_model=clip_vit_B16_model, clip_vit_L14_model=clip_vit_L14_model,
        albef_model = albef_model, albef_txt_processors=albef_txt_processors,
        #albef_vis_processors=albef_vis_processors,
        blip_model_captioning=blip_model_captioning, blip_model_feature_extractor=blip_model_feature_extractor, 
        coca_model=coca_model, coca_tokenizer=coca_tokenizer,
        #coca_transform=coca_transform,
        beit3_model = beit3_model, beit3_tokenizer=beit3_tokenizer, 
        align_model=align_model, align_processor=align_processor, 
        siglip_tokenizer=siglip_tokenizer, siglip_vision_model=siglip_vision_model, siglip_text_model=siglip_text_model, 
        #siglip_processor=siglip_processor, 
        device=device)

In [ ]:
avg_clip_rn_101_score = 0.0
avg_clip_vit_b16_score = 0.0
avg_clip_vit_b32_score = 0.0
avg_clip_vit_l14_score = 0.0

begin_samples = 0
end_samples = 1000
num_samples = end_samples - begin_samples

ori_dir = '/kaggle/input/1k-images/original_images'
target_dir = '/kaggle/input/1k-images/target_images'

score_path = '/kaggle/working/score.txt'
os.makedirs(os.path.dirname(score_path), exist_ok=True)

image_filenames = sorted(os.listdir(ori_dir))[begin_samples:end_samples]

for filename in image_filenames:

    '''
    print("First phase")
    print_memory_usage('cuda:0')
    if torch.cuda.device_count() > 1:
        print_memory_usage('cuda:1')
    '''
     
    ori_image_path = os.path.join(ori_dir, filename)
    target_image_path = os.path.join(target_dir, filename)

    ori_image = Image.open(ori_image_path)
    ori_image = ori_image.resize((224, 224))

    target_image = Image.open(target_image_path)

    name_without_ext = os.path.splitext(filename)[0]
    target_caption = name_without_ext.replace('_', ' ')
    print("target caption: ", target_caption)

    del name_without_ext

    ori_image_tensor = T.ToTensor()(ori_image)
    ori_image_tensor = ori_image_tensor.unsqueeze(0)

    tgt_image_tensor = T.ToTensor()(target_image)
    tgt_image_tensor = tgt_image_tensor.unsqueeze(0)

    transform = transforms.Compose([
        transforms.Resize((512, 512)),  
    ])
    resized_ori_image = transform(ori_image)

    preprocess_image = transforms.Compose([
        transforms.ToTensor()])

    resized_ori_image = preprocess_image(resized_ori_image)
    resized_ori_image = (resized_ori_image * 255).to(torch.uint8)
    ori_text = ofa_caption_tensor_image(resized_ori_image)
    ori_text = ori_text[0]
    #print("original caption: ", ori_text)
    del resized_ori_image
        
    torch.cuda.empty_cache()
    gc.collect()

    '''
    print("End first phase")
    print_memory_usage('cuda:0')
    if torch.cuda.device_count() > 1:
        print_memory_usage('cuda:1')
    '''
    with torch.no_grad():
        ori_image_embedding_clip_vit_b32, ori_text_embedding_clip_vit_b32, target_image_embedding_clip_vit_b32, target_text_embedding_clip_vit_b32, ori_image_embedding_clip_rn_101, ori_text_embedding_clip_rn_101, target_image_embedding_clip_rn_101, target_text_embedding_clip_rn_101, ori_image_embedding_clip_vit_b16, ori_text_embedding_clip_vit_b16, target_image_embedding_clip_vit_b16, target_text_embedding_clip_vit_b16, ori_image_embedding_clip_vit_l14,  ori_text_embedding_clip_vit_l14, target_image_embedding_clip_vit_l14, target_text_embedding_clip_vit_l14, ori_image_embedding_albef, ori_text_embedding_albef, target_image_embedding_albef, target_text_embedding_albef, ori_image_embedding_blip, ori_text_embedding_blip, target_image_embedding_blip, target_text_embedding_blip, ori_image_embedding_coca, ori_text_embedding_coca, target_image_embedding_coca, target_text_embedding_coca, ori_image_embedding_beit3, ori_text_embedding_beit3, target_image_embedding_beit3, target_text_embedding_beit3, ori_image_embedding_align, ori_text_embedding_align, target_image_embedding_align, target_text_embedding_align, ori_image_embedding_siglip, ori_text_embedding_siglip, target_image_embedding_siglip, target_text_embedding_siglip = attack_model.get_full_embedding(ori_image, ori_text, target_image, target_caption)

        tokens = clip.tokenize([target_caption])
        tgt_clip_rn_101_embedding = clip_rn_101_model.encode_text(tokens.to("cuda:1")).to("cpu")
        tgt_clip_vit_b16_embedding = clip_vit_B16_model.encode_text(tokens.to("cuda:1")).to("cpu")
        tgt_clip_vit_b32_embedding = clip_vit_B32_model.encode_text(tokens.to("cuda:1")).to("cpu")
        tgt_clip_vit_l14_embedding = clip_vit_L14_model.encode_text(tokens.to("cuda:0")).to("cpu")

        tgt_clip_rn_101_embedding = tgt_clip_rn_101_embedding / tgt_clip_rn_101_embedding.norm(dim=1, keepdim=True)
        tgt_clip_vit_b16_embedding = tgt_clip_vit_b16_embedding / tgt_clip_vit_b16_embedding.norm(dim=1, keepdim=True)
        tgt_clip_vit_b32_embedding = tgt_clip_vit_b32_embedding / tgt_clip_vit_b32_embedding.norm(dim=1, keepdim=True)
        tgt_clip_vit_l14_embedding = tgt_clip_vit_l14_embedding / tgt_clip_vit_l14_embedding.norm(dim=1, keepdim=True)

    del tokens
    torch.cuda.empty_cache()

    '''
    print("Second phase")
    print_memory_usage('cuda:0')
    if torch.cuda.device_count() > 1:
        print_memory_usage('cuda:1')
    '''

    clip_rn_101_score, clip_vit_b16_score, clip_vit_b32_score, clip_vit_l14_score = evaluate_one_solution([1, 1, 1, 0, 1, 1, 1, 1, 1, 1])
    avg_clip_rn_101_score += clip_rn_101_score
    avg_clip_vit_b16_score += clip_vit_b16_score
    avg_clip_vit_b32_score += clip_vit_b32_score
    avg_clip_vit_l14_score += clip_vit_l14_score

    with open(score_path, 'a') as f_score:
        f_score.write(f"{clip_rn_101_score} {clip_vit_b16_score} {clip_vit_b32_score} {clip_vit_l14_score}\n")

    del clip_rn_101_score, clip_vit_b16_score, clip_vit_b32_score, clip_vit_l14_score
    
    del ori_image, target_image, ori_image_tensor
    del ori_image_embedding_clip_vit_b32, ori_image_embedding_clip_rn_101, ori_image_embedding_clip_vit_b16, ori_image_embedding_clip_vit_l14, ori_image_embedding_albef, ori_image_embedding_blip, ori_image_embedding_coca, ori_image_embedding_beit3, ori_image_embedding_align, ori_image_embedding_siglip, ori_text_embedding_clip_vit_b32, ori_text_embedding_clip_rn_101, ori_text_embedding_clip_vit_b16, ori_text_embedding_clip_vit_l14, ori_text_embedding_albef, ori_text_embedding_blip, ori_text_embedding_coca, ori_text_embedding_beit3, ori_text_embedding_align, ori_text_embedding_siglip, target_image_embedding_clip_vit_b32, target_image_embedding_clip_rn_101, target_image_embedding_clip_vit_b16, target_image_embedding_clip_vit_l14, target_image_embedding_albef, target_image_embedding_blip, target_image_embedding_coca, target_image_embedding_beit3, target_image_embedding_align, target_image_embedding_siglip, target_text_embedding_clip_vit_b32, target_text_embedding_clip_rn_101, target_text_embedding_clip_vit_b16, target_text_embedding_clip_vit_l14, target_text_embedding_albef, target_text_embedding_blip, target_text_embedding_coca, target_text_embedding_beit3, target_text_embedding_align, target_text_embedding_siglip 
    del tgt_clip_rn_101_embedding, tgt_clip_vit_b16_embedding, tgt_clip_vit_b32_embedding, tgt_clip_vit_l14_embedding
    torch.cuda.empty_cache()
    gc.collect()

    '''
    print("Final phase")
    print_memory_usage('cuda:0')
    if torch.cuda.device_count() > 1:
        print_memory_usage('cuda:1')
    '''

avg_clip_rn_101_score /= num_samples
avg_clip_vit_b16_score /= num_samples
avg_clip_vit_b32_score /= num_samples
avg_clip_vit_l14_score /= num_samples

In [36]:
del attack_model
torch.cuda.empty_cache()

In [ ]:
print("avg clip rn 101 score: ", avg_clip_rn_101_score)
print("avg clip vit b16 score: ", avg_clip_vit_b16_score)
print("avg clip vit b32 score: ", avg_clip_vit_b32_score)
print("avg clip vit l14 score: ", avg_clip_vit_l14_score)

print('unified metric: ', avg_clip_rn_101_score * avg_clip_vit_b16_score * avg_clip_vit_b32_score * avg_clip_vit_l14_score)

del avg_clip_rn_101_score, avg_clip_vit_b16_score, avg_clip_vit_b32_score, avg_clip_vit_l14_score